# 03 Active Neuroimaging Diagnostic Agent Baseline

Minimal end-to-end baseline using cached frozen NeuroVFM tokens.

```text
cached NeuroVFM MRI tokens
→ discrete 3D patch environment
→ move / inspect / diagnose actions
→ anomaly evidence memory
→ PPO policy
→ AD/CN prediction + explored-region count
```

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd().resolve()
if not (REPO / "src").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import yaml
import matplotlib.pyplot as plt

from env.data import build_token_manifest, load_balanced_samples
from env.brain_patch_env import BrainPatchEnv, N_ACTIONS
from models.anomaly import DistanceAnomalyDetector
from training.ppo import PPOConfig, train_ppo
from evaluation.active_agent import run_episode, evaluate_policy
from evaluation.brain_space import plot_mri_trajectory_3d_interactive

cfg = yaml.safe_load(open(REPO / "configs" / "active_neuro_agent.yaml"))
rng = np.random.default_rng(cfg["seed"])
cfg

## 1. Load a Small Real Token Dataset

Labels come from the existing OASIS fold CSV in `../brainet`. Token files are cached frozen NeuroVFM outputs, so this notebook does not rerun NeuroVFM.

In [ ]:
manifest = build_token_manifest(
    REPO / cfg["folds_csv"],
    REPO / cfg["token_dir"],
    REPO / cfg["metadata_csv"],
    fold=cfg["fold"],
    split="train",
)
print("matched token rows:", len(manifest))
print(manifest["class_name"].value_counts())

samples = load_balanced_samples(manifest, n_per_class=cfg["n_per_class"], seed=cfg["seed"])
print("loaded samples:", len(samples))
for sample in samples[:4]:
    print(sample["sample_id"], sample["class_name"], sample["embeddings"].shape, sample["coords"].shape, sample["grid_shape"])

## 2. Fit a Minimal Anomaly Detector

The first anomaly detector is deliberately simple: fit a CN reference centroid, then score each token by z-scored distance from that centroid.

In [ ]:
cn_tokens = np.concatenate([s["embeddings"] for s in samples if s["label"] == 0], axis=0)
anomaly_detector = DistanceAnomalyDetector().fit(cn_tokens)

sample = samples[0]
scores = anomaly_detector.score(sample["embeddings"])
print("sample:", sample["sample_id"], sample["class_name"])
print("tokens:", sample["embeddings"].shape[0])
print("embedding dim:", sample["embeddings"].shape[1])
print("coords range:", sample["coords"].min(axis=0).tolist(), sample["coords"].max(axis=0).tolist())
print("anomaly score min/mean/max:", float(scores.min()), float(scores.mean()), float(scores.max()))

## 3. Run One Episode

Actions: `0..5` move in the 3D token grid, `6` inspect, `7` diagnose CN, `8` diagnose AD.

In [ ]:
env_kwargs = cfg["environment"]
env = BrainPatchEnv(sample, anomaly_detector, seed=cfg["seed"], **env_kwargs)
obs = env.reset()
print("obs dim:", obs.shape)
print("n actions:", N_ACTIONS)

# Tiny hand-written rollout: inspect twice, move once, then stop.
for action in [6, 0, 6, 8]:
    obs, reward, done, info = env.step(action)
    print("action:", action, "reward:", round(reward, 3), "done:", done, info)
    if done:
        break

print("label:", sample["class_name"], "prediction:", env.trace.prediction)
print("inspected:", len(env.trace.inspected_indices), "visited:", len(env.trace.visited_indices))

## 4. Train a Minimal PPO Baseline

This trains only the policy/value network. NeuroVFM tokens are cached and frozen.

In [ ]:
def make_env():
    s = samples[int(rng.integers(0, len(samples)))]
    return BrainPatchEnv(s, anomaly_detector, seed=int(rng.integers(0, 1_000_000)), **env_kwargs)

probe_env = make_env()
ppo_cfg = PPOConfig(
    epochs=cfg["ppo"]["epochs"],
    episodes_per_update=cfg["ppo"]["episodes_per_update"],
    gamma=cfg["ppo"]["gamma"],
    clip_ratio=cfg["ppo"]["clip_ratio"],
    lr=cfg["ppo"]["lr"],
)
policy, history = train_ppo(
    make_env,
    obs_dim=probe_env.obs_dim,
    n_actions=N_ACTIONS,
    updates=cfg["ppo"]["updates"],
    config=ppo_cfg,
    seed=cfg["seed"],
)
history

## 5. Evaluate Accuracy and Exploration Cost

In [ ]:
def eval_envs():
    return [BrainPatchEnv(s, anomaly_detector, seed=100 + i, **env_kwargs) for i, s in enumerate(samples)]

metrics = evaluate_policy(eval_envs, policy)
metrics

## 6. Visualize Visited Trajectory and Anomaly Scores

In [ ]:
vis_sample = samples[0]
vis_env = BrainPatchEnv(vis_sample, anomaly_detector, seed=123, **env_kwargs)
trace = run_episode(vis_env, policy=policy, greedy=True)

# Early PPO baselines may stop immediately. For visualization only,
# fall back to a scripted inspection rollout if no evidence was inspected.
if len(trace["inspected_indices"]) == 0:
    vis_env = BrainPatchEnv(vis_sample, anomaly_detector, seed=123, **env_kwargs)
    obs = vis_env.reset()
    for action in [6, 0, 6, 2, 6, 8]:
        obs, reward, done, info = vis_env.step(action)
        if done:
            break
    trace = {
        "label": vis_env.label,
        "prediction": vis_env.trace.prediction,
        "visited_indices": vis_env.trace.visited_indices,
        "inspected_indices": vis_env.trace.inspected_indices,
        "anomaly_scores": vis_env.trace.anomaly_scores,
        "actions": vis_env.trace.actions,
        "rewards": vis_env.trace.rewards,
    }

coords = vis_sample["coords"]
all_scores = anomaly_detector.score(vis_sample["embeddings"])
visited = np.asarray(trace["visited_indices"], dtype=int)
inspected = np.asarray(trace["inspected_indices"], dtype=int)
html_path = REPO / "outputs" / "active_agent_trajectory_mri_space_interactive.html"

# Interactive MRI-space view: rotate, zoom, and pan in the notebook.
# Token→voxel mapping replays NeuroVFM's preprocessing geometry on volume_path
# (must be the exact file the tokens were extracted from; grid_shape is checked).
fig, voxel_coords = plot_mri_trajectory_3d_interactive(
    volume_path=vis_sample["volume_path"],
    token_coords=coords,
    grid_shape=vis_sample["grid_shape"],
    anomaly_scores=all_scores,
    visited_indices=visited,
    inspected_indices=inspected,
    title=f"MRI-space trajectory: {vis_sample['sample_id']} ({vis_sample['class_name']})",
    html_path=html_path,
    stride=3,
)
fig.show()

print("interactive html:", html_path)
print("true label:", trace["label"], "prediction:", trace["prediction"])
print("actions:", trace["actions"])
print("visited regions:", len(trace["visited_indices"]))
print("inspected regions:", len(trace["inspected_indices"]))
print("visited token coords [D,H,W]:", coords[visited].tolist() if len(visited) else [])
print("visited MRI voxel coords:", voxel_coords[visited].round(1).tolist() if len(visited) else [])
print("volume path:", vis_sample["volume_path"])
